# Laborator 11 - Algoritmi evolutivi - Delfini

Se cere identificarea comunităților existente într-o rețea folosind:
- un algoritm predefinit intr-o biblioteca specializata (e.g. networkx, gephi, altele);
- un algoritm evolutiv.

Metodologia pentru rezolvarea unei probleme cu ajutorul algoritmilor evolutivi
- analiza problemei si reprezentarea potentialelor solutii
- definirea functiei de fitness
- definirea operatorilor genetici (Selectie, incrucisare, mutatie)
- stabilirea schemei evolutive (algoritm generational, steady-state, etc.)
- alegerea solutiei optime

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import networkx as nx
from networkx.linalg.graphmatrix import adjacency_matrix
from pyvis.network import Network

### Vizualizarea retelei

In [2]:
# graful care descrie reteaua
G = nx.read_gml("../../given_networks/dolphins/dolphins.gml")

labelList = list(G.nodes())
adjacencyMatrix = nx.to_numpy_array(G).tolist()
adjacencyMatrix = [[int(x) for x in row] for row in adjacencyMatrix]


# se pregateste vectorul de culori -> ca sa stie programul cum sa coloreze fiecare nod in parte
generalColors = ["red","green","yellow","blue","magenta","cyan","orange"]
colors = []
with open("../../given_networks/dolphins/classLabeldolphins.txt", "r") as f:
    for linie in f:
        date = linie.strip().split()
        if len(date) == 2:
            id_text = int(date[0])
            community = int(date[1])

            name = labelList[id_text-1]
            G.nodes[name]["group"] = community

            colors.append(generalColors[community - 1])


'''
plt.figure(figsize=(12, 12))

# este necesara functia de layout -> calculeaza pozitiile fizice ale comunitatilor
layoutFunction = nx.spring_layout(G)
nx.draw_networkx_nodes(G,layoutFunction,node_color=colors)
nx.draw_networkx_edges(G,layoutFunction)

# afisarea efectiva in fereastra
plt.title("Dolphins - True Community Network")
plt.show()
'''
pos = nx.spring_layout(G, seed=42)

for node in G.nodes():
    G.nodes[node]['x'] = pos[node][0] * 1000
    G.nodes[node]['y'] = pos[node][1] * 1000

network = Network(notebook=True,bgcolor='#222222',font_color='white')
network.from_nx(G)
network.show("dolphins_true.html")


dolphins_true.html


### Determinarea comunitatilor cu TOOL

In [3]:
from networkx.algorithms import community

# se introduce si param seed, deoarece, in cadrul algoritmului exista acea comp. aleatorie
# se vor returna seturi (comunitatile) ce contin id-urile respectivilor delfini
communityTOOL = community.louvain_communities(G, seed=11)

# reprezentarea numerica
nameToId = {name: idx for idx, name in enumerate(G.nodes())}
communityTOOLNumerical = []

for nodeSet in communityTOOL:
    # se transforma fiecare nume in id-ul sau
    currentCommunity = [nameToId[nodeName] for nodeName in nodeSet]
    communityTOOLNumerical.append(currentCommunity)


for idComunity, com in enumerate(communityTOOL):
    for nod in com:
        G.nodes[nod]['group'] = idComunity

network = Network(notebook=True,bgcolor='#222222',font_color='white')
network.from_nx(G)
network.show("dolphins_tool.html")

dolphins_tool.html


### Evaluarea performantei -> scorul de modularitate

In [4]:
def modularity(communities, adjacencyMatrix):
    m = sum(sum(row) for row in adjacencyMatrix) / 2
    Q = 0
    for community in communities:
        for i in community:
            for j in community:
                ki = sum(adjacencyMatrix[i])
                kj = sum(adjacencyMatrix[j])

                Q += adjacencyMatrix[i][j] - ki * kj / (2 * m)

    Q *= 1/(2*m)
    return Q

print(modularity(communityTOOLNumerical, adjacencyMatrix))

0.5241090146750512


### Determinarea comunitatilor folosind cod propriu

In [5]:
from fitnessModularity import fitnessFunction
import functools
from GeneticAlgorithm import GA

fitnessFunc = functools.partial(fitnessFunction, adjacencyMatrix=adjacencyMatrix)

problParam = {
    'adjacencyMatrix': adjacencyMatrix,
    'function': fitnessFunc
}

param = {
    'popSize': len(G.nodes()),
    'noGen': 300,
    'mutProb': 0.1
}

ga = GA(param=param, problParam=problParam)
best = ga.run()
decodedBest = best.decode()


# se seteaza noile comunitati determinate in graf
nodesList = list(G.nodes())
for communityId, nodesInCommunity in enumerate(decodedBest):
    for nodeIndex in nodesInCommunity:
        nodeName = nodesList[nodeIndex]
        G.nodes[nodeName]['group'] = communityId


network = Network(notebook=True, bgcolor='#222222', font_color='white')
network.from_nx(G)
network.show("dolphins_propriu.html")

dolphins_propriu.html


### Evaluarea performantei -> scorul de modularitate

In [6]:
print(modularity(decodedBest,adjacencyMatrix))

0.43874846722835287
